In [ ]:
import re
import json
import statistics
from collections import defaultdict
from dataclasses import dataclass, field, asdict
from dotenv import load_dotenv
import os

## Carrega variáveis do .env

In [ ]:
load_dotenv()

## Categorizar Log

In [ ]:
# ---------------------------------------------------------------------------
# Linha base de todo evento estruturado
# ---------------------------------------------------------------------------
LINHAS_BASE = re.compile(
    r"^(?P<ts>\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2}) "
    r"(?P<level>INFO|WARN|ERROR|DEBUG) "
    r"(?P<component>[\w$.]+): "
    r"(?P<msg>.*)$"
)

# ---------------------------------------------------------------------------
# Máscara de dados sensíveis - host/IP e paths s3a:// 
# ---------------------------------------------------------------------------
HOST_PATTERN = re.compile(r"\[?([0-9a-fA-F:\.]{7,})\]?(?::\d+)?")
PATH_PATTERN = re.compile(r"s3a?://[^\s,)\"]+")


class Sanitizador:
    def __init__(self):
        self.host_map: dict[str, str] = {}
        self.path_map: dict[str, str] = {}

    def mask_host(self, host: str) -> str:
        m = re.match(r"\[?([0-9a-fA-F:\.]+?)\]?(?::\d+)?$", host)
        normalized = m.group(1) if m else host
        if normalized not in self.host_map:
            self.host_map[normalized] = f"host_{len(self.host_map) + 1}"
        return self.host_map[normalized]

    def mask_paths(self, text: str) -> str:
        def _replace(m):
            raw = m.group(0)
            if raw not in self.path_map:
                self.path_map[raw] = f"path_{len(self.path_map) + 1}"
            return self.path_map[raw]
        return PATH_PATTERN.sub(_replace, text)

    def mask_text(self, text: str) -> str:
        """Mascara path e qualquer IPv4/IPv6 solto numa mensagem crua
        (usado em raw_warn_error e em linhas de continuação de stack trace)."""
        text = self.mask_paths(text)

        def _replace_host(m):
            return f"[{self.mask_host(m.group(1))}]"
        return HOST_PATTERN.sub(_replace_host, text)


# ---------------------------------------------------------------------------
# Padrões de evento
# ---------------------------------------------------------------------------
PADROES_DE_EVENTOS = {
    "job_start": re.compile(r"^Got job (?P<job_id>\d+) \((?P<action>.*?)\) with (?P<partitions>\d+) output partitions$"),
    "job_finish": re.compile(r"^Job (?P<job_id>\d+) finished: .*?, took (?P<duration_s>[\d.]+) s$"),
    "stage_submit": re.compile(r"^Submitting (?P<stage>ResultStage|ShuffleMapStage) (?P<stage_id>\d+)"),
    "stage_finish": re.compile(r"^(?P<stage>ResultStage|ShuffleMapStage) (?P<stage_id>\d+) \(.*?\) finished in (?P<duration_s>[\d.]+) s$"),
    "taskset_removed": re.compile(r"^Removed TaskSet (?P<stage_id>\d+)\.(?P<attempt>\d+), whose tasks have all completed"),
    "stage_failed": re.compile(r"^(?P<stage>ResultStage|ShuffleMapStage) (?P<stage_id>\d+) \(.*?\) failed"),
    "task_start": re.compile(
        r"^Starting task (?P<task_id>[\d.]+) in stage (?P<stage_id>[\d.]+) \(TID (?P<tid>\d+)\) "
        r"\((?P<host>[^,]+), executor (?P<executor>\d+)"
    ),
    "task_finish": re.compile(
        r"^Finished task (?P<task_id>[\d.]+) in stage (?P<stage_id>[\d.]+) \(TID (?P<tid>\d+)\) "
        r"in (?P<duration_ms>\d+) ms on (?P<host>[^\s(]+) \(executor (?P<executor>\d+)\)"
    ),
    "broadcast_stored": re.compile(
        r"^Block broadcast_(?P<broadcast_id>\d+)(?:_piece\d+)? stored as (?:values|bytes) in memory "
        r"\(estimated size (?P<size_val>[\d.]+) (?P<size_unit>\w+),(?: actual size: [\d.]+ \w+,)? "
        r"free (?P<free_val>[\d.]+) (?P<free_unit>\w+)\)$"
    ),
    "block_added": re.compile(
        r"^Added broadcast_(?P<broadcast_id>\d+)_piece\d+ in memory on "
        r"(?P<host>\[[0-9a-fA-F:\.]+\]:\d+) "
        r"\(size: (?P<size_val>[\d.]+) (?P<size_unit>\w+), free: (?P<free_val>[\d.]+) (?P<free_unit>\w+)\)$"
    ),
    "shuffle_map_output_request": re.compile(r"^Asked to send map output locations for shuffle (?P<shuffle_id>\d+)$"),
    "shuffle_partition_advisory": re.compile(r"^For shuffle\((?P<shuffle_id>\d+)\), advisory target size: (?P<advisory_bytes>\d+)"),
    "executor_backlog_request": re.compile(r"^Requesting (?P<num_requested>\d+) new executors because tasks are backlogged"),
    "executor_registered": re.compile(r"^New executor (?P<executor>\d+) has registered \(new total is (?P<total>\d+)\)$"),
    "executor_not_found": re.compile(r"^No executor found for (?P<host>[0-9a-fA-F:\.]+)$"),
    "executor_lost": re.compile(r"^[Ee]xecutor (?P<executor>\d+) lost"),
    "app_final_status": re.compile(r"^SparkContext is stopping with exitCode (?P<exit_code>\d+)\.?$"),

}


@dataclass
class LogEvent:
    ts: str
    level: str
    component: str
    event_type: str
    data: dict = field(default_factory=dict)


def parse_log(text: str, sanitizador: Sanitizador | None = None) -> list[LogEvent]:
    if sanitizador is None:
        sanitizador = Sanitizador()

    eventos: list[LogEvent] = []
    pendente_stack_trace: LogEvent | None = None

    for linha_bruta in text.splitlines():
        linha = linha_bruta.rstrip("\n")
        m = LINHAS_BASE.match(linha.strip())

        if not m:

            if pendente_stack_trace is not None and linha.strip():
                pendente_stack_trace.data.setdefault("stack_trace", [])
                pendente_stack_trace.data["stack_trace"].append(
                    sanitizador.mask_text(linha.strip())
                )
            continue

        ts, level, component, msg = m.group("ts", "level", "component", "msg")
        pendente_stack_trace = None

        matched = False
        for event_type, pattern in PADROES_DE_EVENTOS.items():
            em = pattern.match(msg)
            if em:
                data = em.groupdict()
                if "host" in data and data["host"]:
                    data["host_masked"] = sanitizador.mask_host(data.pop("host"))
                evento = LogEvent(ts, level, component, event_type, data)
                eventos.append(evento)
                matched = True
                break

        if not matched and level in ("WARN", "ERROR"):
            evento = LogEvent(ts, level, component, "raw_warn_error",
                               {"msg": sanitizador.mask_text(msg)})
            eventos.append(evento)
            pendente_stack_trace = evento

    return eventos


def agregado_por_stage(eventos: list[LogEvent]) -> list[dict]:
    tasks: dict[str, dict] = {}
    for e in eventos:
        if e.event_type in ("task_start", "task_finish"):
            tasks.setdefault(e.data["tid"], {}).update(e.data)

    by_stage: dict[str, list[dict]] = defaultdict(list)
    for t in tasks.values():
        if "duration_ms" in t and "stage_id" in t:
            stage_id = t["stage_id"].split(".")[0]
            by_stage[stage_id].append(t)

 
    stage_meta = {}
    for e in eventos:
        if e.event_type == "stage_finish":
            stage_meta[e.data["stage_id"]] = {"duration_s": float(e.data["duration_s"]), "fonte": "stage_finish"}
        elif e.event_type == "taskset_removed" and e.data["stage_id"] not in stage_meta:
            stage_meta[e.data["stage_id"]] = {"duration_s": None, "fonte": "taskset_removed (sem duração precisa)"}

    summary = []
    for stage_id, task_list in by_stage.items():
        durations = [int(t["duration_ms"]) for t in task_list]
        per_executor = defaultdict(list)
        for t in task_list:
            per_executor[t.get("executor", "?")].append(int(t["duration_ms"]))

        durations_sorted = sorted(durations)
        p95_idx = max(0, int(len(durations_sorted) * 0.95) - 1)
        median = statistics.median(durations)

        summary.append({
            "stage_id": stage_id,
            "num_tasks": len(task_list),
            "duration_s": stage_meta.get(stage_id, {}).get("duration_s"),
            "duration_fonte": stage_meta.get(stage_id, {}).get("fonte"),
            "task_duration_ms": {
                "min": min(durations),
                "max": max(durations),
                "mean": round(statistics.mean(durations), 1),
                "median": median,
                "p95": durations_sorted[p95_idx],
            },
            "tasks_per_executor": {ex: len(v) for ex, v in per_executor.items()},
            "executor_duration_ms": {
                ex: {"mean": round(statistics.mean(v), 1), "total": sum(v)}
                for ex, v in per_executor.items()
            },
            "skew_ratio": round(max(durations) / median, 2) if median > 0 else None,
        })

    return sorted(summary, key=lambda s: int(s["stage_id"]))


def resumo_eventos_esparsos(eventos: list[LogEvent]) -> dict:
    contagem = defaultdict(int)
    for e in eventos:
        contagem[e.event_type] += 1
    return dict(sorted(contagem.items(), key=lambda kv: -kv[1]))


if __name__ == "__main__":
    path = os.getenv("LOG_REF_TRACKING")

    if not path:
        raise ValueError("A variável não foi encontrada no arquivo .env.")

    if not os.path.isfile(path):
        raise FileNotFoundError(f"Arquivo não encontrado!")

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    sanitizador = Sanitizador()
    parsed = parse_log(content, sanitizador)

    stage_summary = agregado_por_stage(parsed)
    raw_eventos = [asdict(e) for e in parsed if e.event_type == "raw_warn_error"]
    app_status = [asdict(e) for e in parsed if e.event_type == "app_final_status"]

    output = {
        "stages": stage_summary,
        "eventos_esparsos": resumo_eventos_esparsos(parsed),
        "status_final": app_status,
        "raw_warn_error": raw_eventos,
    }

    print(json.dumps(output, indent=2, ensure_ascii=False))

    with open("log_agregado.json", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print("Arquivo 'log_agregado.json' gerado com sucesso.")
    print(f"Hosts mascarados: {len(sanitizador.host_map)} | Paths mascarados: {len(sanitizador.path_map)}")